# Gold Grid Incident Summary

This notebook creates the Gold table:

`gold_grid_incident_summary`

Business goal:
- summarize operational grid incidents
- analyze incident severity by day and region
- support operational monitoring and reporting

Aggregation grain:
- event_day
- region
- severity_band

In [0]:
import yaml
from pyspark.sql import functions as F

# Load project config
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Catalog and schemas
catalog = config["catalog"]              # e.g., vattenfall_dev
silver_schema = "refined"
gold_schema = "gold"                     # target Gold schema

# Fully qualified table names
silver_grid_events = f"{catalog}.{silver_schema}.silver_grid_events"
gold_grid_events = f"{catalog}.{gold_schema}.gold_grid_events"


print(f"Reading source table: {source_table}")

grid_df = spark.table(source_table)

rows_read = grid_df.count()

print(f"Rows read: {rows_read}")

display(grid_df.limit(10))

In [0]:

print(f"Reading source table: {silver_grid_events}")

grid_df = spark.table(silver_grid_events)

rows_read = grid_df.count()

print(f"Rows read: {rows_read}")

display(grid_df.limit(10))

## Create Grid Incident Aggregation

Aggregate incidents by:
- event_day
- region
- severity_band

Business metrics:
- incident count
- total incident duration
- average incident duration
- elevated incident count

In [0]:
gold_grid_df = (
    grid_df
    .groupBy("event_day", "region", "severity_band")
    .agg(
        F.count("event_id").alias("incident_count"),

        F.sum("duration_minutes").alias("total_duration_minutes"),

        F.avg("duration_minutes").alias("avg_duration_minutes"),

        F.sum(
            F.when(
                F.col("severity").isin("HIGH", "MEDIUM"),
                1
            ).otherwise(0)
        ).alias("elevated_incident_count")
    )
)

print("Aggregation grain: event_day + region + severity_band")

print("Business metrics created:")
print("- incident count")
print("- total duration")
print("- average duration")
print("- elevated incident count")

## Create Grid Incident Aggregation

Aggregate incidents by:
- event_day
- region
- severity_band

Business metrics:
- incident count
- total incident duration
- average incident duration
- elevated incident count

In [0]:
gold_grid_df = (
    gold_grid_df
    .groupBy("event_day", "region", "severity_band")
    .agg(
        F.sum("incident_count").alias("incident_count"),
        F.sum("total_duration_minutes").alias("total_duration_minutes"),
        F.avg("avg_duration_minutes").alias("avg_duration_minutes"),

        # ✅ THIS IS THE REQUIRED LOGIC
        F.sum(
            F.when(
                F.col("severity_band").isin("SEVERE", "MODERATE"),
                1
            ).otherwise(0)
        ).alias("elevated_incident_count")
    )
)

print("✅ Elevated incident KPI correctly calculated")

print("Aggregation grain: event_day + region + severity_band")

print("Business metrics created:")
print("- incident count")
print("- total duration")
print("- average duration")
print("- elevated incident count")

## Improve Numeric Readability

Round average duration metrics to improve:
- dashboard readability
- reporting quality
- presentation clarity

In [0]:
gold_grid_df = (
    gold_grid_df
    .withColumn(
        "avg_duration_minutes",
        F.round("avg_duration_minutes", 2)
    )
)

display(gold_grid_df)

## Create Gold Schema

Ensure the Gold schema exists before writing the Gold table.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS vattenfall_dev.gold")

## Define Gold Target Table

Target table:
- `gold_grid_incident_summary`

This Gold table is designed for:
- operational dashboards
- incident monitoring
- severity analysis
- regional operational reporting

In [0]:

target_table = "vattenfall_dev.gold.gold_grid_events"

print(f"Target table: {target_table}")

## Write Gold Table

Write the aggregated grid incident summary
to the Gold layer using Delta format.

In [0]:
(
    gold_grid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

rows_written = gold_grid_df.count()

print(f"Rows written: {rows_written}")

print("✅ gold_grid_incident_summary created successfully")

## Validate Gold Output

Inspect the final Gold dataset to confirm:
- incident aggregation quality
- severity distribution
- duration metrics
- dashboard readiness

In [0]:
display(
    gold_grid_df.limit(5)
)